## Example 1: Messages are protobuf objects with helpers

In [ ]:
import importlib.metadata

from a2a.helpers import get_message_text, new_text_message
from a2a.types import Role
from google.protobuf import json_format

# Example 1: A2A 1.x types are protobuf messages; helpers build and read them
assert importlib.metadata.version("a2a-sdk") == "1.3.0"
message = new_text_message("What is the capital of France?", context_id="ctx-1")
assert message.role == Role.ROLE_AGENT and get_message_text(message) == "What is the capital of France?"
wire = json_format.MessageToDict(message)
assert wire["contextId"] == "ctx-1" and wire["parts"][0]["text"].startswith("What"), wire
print("Example 1 OK:", wire["role"], wire["parts"][0]["text"])

## Example 2: Publish an agent card at the well-known path

In [ ]:
from a2a.server.routes import create_agent_card_routes
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill
from a2a.utils import AGENT_CARD_WELL_KNOWN_PATH
from starlette.applications import Starlette
from starlette.testclient import TestClient

# Example 2: the card a client discovers before talking to an agent
card = AgentCard(name="Echo", description="Repeats what you say", version="1.0.0",
                 supported_interfaces=[AgentInterface(url="http://localhost/", protocol_binding="JSONRPC")],
                 capabilities=AgentCapabilities(streaming=False), default_input_modes=["text/plain"], default_output_modes=["text/plain"],
                 skills=[AgentSkill(id="echo", name="Echo", description="Echo the input", tags=["demo"])])
with TestClient(Starlette(routes=create_agent_card_routes(card))) as http:
    served = http.get(AGENT_CARD_WELL_KNOWN_PATH).json()
assert served["name"] == "Echo" and served["skills"][0]["id"] == "echo", served
print("Example 2 OK:", AGENT_CARD_WELL_KNOWN_PATH, served["supportedInterfaces"])

## Example 3: A full agent round trip over JSON-RPC, in process

In [ ]:
import httpx
from a2a.client import ClientConfig, create_client
from a2a.helpers import get_message_text, new_text_message
from a2a.server.agent_execution import AgentExecutor
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, Role, SendMessageRequest
from starlette.applications import Starlette

class Echo(AgentExecutor):  # Example 3: an agent that answers "echo: <input>"
    async def execute(self, context, event_queue):
        await event_queue.enqueue_event(new_text_message(f"echo: {context.get_user_input()}", context_id=context.context_id))
    async def cancel(self, context, event_queue):
        pass


card = AgentCard(name="Echo", version="1", supported_interfaces=[AgentInterface(url="http://agent/", protocol_binding="JSONRPC")], capabilities=AgentCapabilities(streaming=False))
handler = DefaultRequestHandler(agent_executor=Echo(), task_store=InMemoryTaskStore(), agent_card=card)
app = Starlette(routes=create_agent_card_routes(card) + create_jsonrpc_routes(handler, rpc_url="/"))
client = await create_client(card, ClientConfig(streaming=False, httpx_client=httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://agent")))
replies = [r async for r in client.send_message(SendMessageRequest(message=new_text_message("hello", role=Role.ROLE_USER)))]
assert get_message_text(replies[-1].message) == "echo: hello", replies
await client.close(); print("Example 3 OK:", get_message_text(replies[-1].message))

## Example 4: Tasks and their store

In [ ]:
from a2a.helpers import new_task_from_user_message, new_text_message
from a2a.server.context import ServerCallContext
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import Role, TaskState

# Example 4: a task made from a user's message is stored and read back by id
task = new_task_from_user_message(new_text_message("summarize this", role=Role.ROLE_USER))
assert task.id and task.context_id and task.status.state == TaskState.TASK_STATE_SUBMITTED, task.status
store, call = InMemoryTaskStore(), ServerCallContext()
await store.save(task, call)
loaded = await store.get(task.id, call)
assert loaded is not None and loaded.id == task.id and loaded.history[0].parts[0].text == "summarize this"
print("Example 4 OK:", task.id, TaskState.Name(task.status.state))

## Example 5: Artifacts carry an agent's results

In [ ]:
from a2a.helpers import get_artifact_text, new_data_part, new_text_artifact
from google.protobuf import json_format

# Example 5: a text artifact, and a structured data part
artifact = new_text_artifact(name="report", text="All 12 checks passed.", description="final report")
assert artifact.name == "report" and get_artifact_text(artifact) == "All 12 checks passed."
part = new_data_part({"passed": 12, "failed": 0})
assert json_format.MessageToDict(part.data) == {"passed": 12, "failed": 0}, part
print("Example 5 OK:", artifact.artifact_id, json_format.MessageToDict(part.data))